# import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Specify the folder path where your text files are stored
folder_path = "/kaggle/input/typology/Typology"

In [ ]:
# Initialize empty lists to store articles and labels
articles = []
labels = []

In [ ]:
# Loop through each file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt"):
        # Read the content and label from the file
        with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as file:
            content = file.read()
            
            # Extract the label from the content
            label_start = content.find("Political Typology:") + len("Political Typology:")
            label = content[label_start:].strip()
            content = content[:label_start].strip()
            
            articles.append(content)
            labels.append(label)

In [ ]:
# Convert labels to binary representation using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(labels)

In [ ]:
# Tokenize and pad sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(articles)
X = tokenizer.texts_to_sequences(articles)
X_padded = pad_sequences(X, padding='post')

In [ ]:
# Convert data to PyTorch tensors
X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.float32)

In [ ]:
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

In [ ]:
# Hyperparameters
embedding_dim = 100
hidden_dim = 128
output_dim = len(mlb.classes_)  # Number of unique political typologies
num_epochs = 5
batch_size = 32
learning_rate = 0.001

In [ ]:
# Model definition
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]  # Take the last output from the sequence
        output = self.fc(lstm_out)
        output = self.sigmoid(output)
        return output

In [ ]:
# Instantiate the model
vocab_size = len(tokenizer.word_index) + 1
model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)

In [ ]:
# Loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Convert data to DataLoader
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

In [ ]:
# Training loop
tqdm_epoch = tqdm(range(num_epochs), desc="Epochs")
for epoch in tqdm_epoch:
    tqdm_batch = tqdm(train_loader, desc="Batch")
    for inputs, labels in tqdm_batch:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        tqdm_batch.set_postfix({"Loss": loss.item()})
    tqdm_epoch.set_postfix({"Final Loss": loss.item()})

In [ ]:
# Evaluate the model on the test set
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    y_pred = model(X_test)
    test_loss = criterion(y_pred, y_test)
    print(f'Test Loss: {test_loss.item()}')

# Save the model if needed
# Specify the path where you want to save the model
model_save_path = 'path/to/save/model.pth'
torch.save(model.state_dict(), model_save_path)
print(f'Model saved at: {model_save_path}')

In [ ]:
# Load the saved model
model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
model.load_state_dict(torch.load(model_save_path))
model.eval()  # Set the model to evaluation mode